In [1]:
import pandas as pd

social_media_data = pd.read_csv('../datasets/social_media_advertisements.csv')


agg_df = social_media_data.groupby(["X", "Z", "Y"]).size().reset_index()

agg_df


,X,Z,Y,0
0,no_social,ad,no_purchase,1
1,no_social,ad,purchase,19
2,no_social,no_ad,no_purchase,38
3,no_social,no_ad,purchase,342
4,social,ad,no_purchase,323
5,social,ad,purchase,57
6,social,no_ad,no_purchase,18
7,social,no_ad,purchase,2


# Front-Door Criterion for Causal Inference

## Problem Setup
We want to estimate the causal effect of X on Y, but there's an **unmeasured confounder** that affects both X and Y (but not Z).

**Causal Structure:**
- X → Z → Y (Z is a mediator)
- U → X, U → Y (U is unmeasured confounder)
- U ⊥ Z (confounder doesn't affect mediator)

## Front-Door Adjustment Formula
When the front-door criterion is satisfied, we can identify the causal effect using:

**P(Y=y|do(X=x)) = Σ_z P(Z=z|X=x) × Σ_{x'} P(Y=y|X=x',Z=z) × P(X=x')**

This formula works by:
1. First step: X → Z (no confounding since we condition on X)
2. Second step: Z → Y (confounding blocked by summing over all X values)

## Requirements for Front-Door Criterion
1. Z intercepts all directed paths from X to Y
2. No unblocked backdoor path from X to Z
3. All backdoor paths from Z to Y are blocked by X

In [2]:
# Front-door criterion estimation of causal effect of X on Y
# Formula: P(Y=y|do(X=x)) = Σ_z P(Z=z|X=x) * Σ_x' P(Y=y|X=x',Z=z) * P(X=x')

import numpy as np

# First, let's examine the data structure
print("Data structure:")
print(agg_df)
print(f"\nTotal observations: {agg_df[0].sum()}")

# Rename the count column for clarity
agg_df = agg_df.rename(columns={0: 'count'})

# Calculate total sample size
N = agg_df['count'].sum()

# Step 1: Calculate P(Z=z|X=x) - effect of X on mediator Z
print("\n=== Step 1: P(Z=z|X=x) ===")
p_z_given_x = agg_df.groupby(['X', 'Z'])['count'].sum().reset_index()
p_z_given_x['total_x'] = p_z_given_x.groupby('X')['count'].transform('sum')
p_z_given_x['p_z_given_x'] = p_z_given_x['count'] / p_z_given_x['total_x']
print(p_z_given_x[['X', 'Z', 'p_z_given_x']])

# Step 2: Calculate P(X=x') - marginal distribution of X
print("\n=== Step 2: P(X=x') ===")
p_x = agg_df.groupby('X')['count'].sum().reset_index()
p_x['p_x'] = p_x['count'] / N
print(p_x[['X', 'p_x']])

# Step 3: Calculate P(Y=y|X=x',Z=z) - effect of X and Z on Y
print("\n=== Step 3: P(Y=y|X=x',Z=z) ===")
p_y_given_xz = agg_df.groupby(['X', 'Z', 'Y'])['count'].sum().reset_index()
p_y_given_xz['total_xz'] = p_y_given_xz.groupby(['X', 'Z'])['count'].transform('sum')
p_y_given_xz['p_y_given_xz'] = p_y_given_xz['count'] / p_y_given_xz['total_xz']
print(p_y_given_xz[['X', 'Z', 'Y', 'p_y_given_xz']])

Data structure:
           X      Z            Y    0
0  no_social     ad  no_purchase    1
1  no_social     ad     purchase   19
2  no_social  no_ad  no_purchase   38
3  no_social  no_ad     purchase  342
4     social     ad  no_purchase  323
5     social     ad     purchase   57
6     social  no_ad  no_purchase   18
7     social  no_ad     purchase    2

Total observations: 800

=== Step 1: P(Z=z|X=x) ===
           X      Z  p_z_given_x
0  no_social     ad         0.05
1  no_social  no_ad         0.95
2     social     ad         0.95
3     social  no_ad         0.05

=== Step 2: P(X=x') ===
           X  p_x
0  no_social  0.5
1     social  0.5

=== Step 3: P(Y=y|X=x',Z=z) ===
           X      Z            Y  p_y_given_xz
0  no_social     ad  no_purchase          0.05
1  no_social     ad     purchase          0.95
2  no_social  no_ad  no_purchase          0.10
3  no_social  no_ad     purchase          0.90
4     social     ad  no_purchase          0.85
5     social     ad     purcha

In [3]:
# Front-door adjustment formula implementation
def front_door_adjustment(x_val, y_val):
    """
    Calculate P(Y=y_val|do(X=x_val)) using front-door criterion
    """
    total_prob = 0
    
    # Get unique values of Z
    z_values = agg_df['Z'].unique()
    x_values = agg_df['X'].unique()
    
    for z in z_values:
        # P(Z=z|X=x_val)
        p_z_x = p_z_given_x[(p_z_given_x['X'] == x_val) & (p_z_given_x['Z'] == z)]
        if len(p_z_x) == 0:
            p_z_x_val = 0
        else:
            p_z_x_val = p_z_x['p_z_given_x'].iloc[0]
        
        # Sum over all x' values: P(Y=y_val|X=x',Z=z) * P(X=x')
        inner_sum = 0
        for x_prime in x_values:
            # P(Y=y_val|X=x',Z=z)
            p_y_xz = p_y_given_xz[(p_y_given_xz['X'] == x_prime) & 
                                  (p_y_given_xz['Z'] == z) & 
                                  (p_y_given_xz['Y'] == y_val)]
            if len(p_y_xz) == 0:
                p_y_xz_val = 0
            else:
                p_y_xz_val = p_y_xz['p_y_given_xz'].iloc[0]
            
            # P(X=x')
            p_x_prime = p_x[p_x['X'] == x_prime]['p_x'].iloc[0]
            
            inner_sum += p_y_xz_val * p_x_prime
        
        total_prob += p_z_x_val * inner_sum
    
    return total_prob

# Calculate causal effects for all combinations
print("\n=== Front-door Causal Effects: P(Y=y|do(X=x)) ===")
results = []

x_values = sorted(agg_df['X'].unique())
y_values = sorted(agg_df['Y'].unique())

for x in x_values:
    for y in y_values:
        causal_effect = front_door_adjustment(x, y)
        results.append({'X': x, 'Y': y, 'P(Y|do(X))': causal_effect})
        print(f"P(Y={y}|do(X={x})) = {causal_effect:.4f}")

# Convert to DataFrame for easier analysis
causal_effects_df = pd.DataFrame(results)
print("\nCausal Effects DataFrame:")
print(causal_effects_df)


=== Front-door Causal Effects: P(Y=y|do(X=x)) ===
P(Y=no_purchase|do(X=no_social)) = 0.4975
P(Y=purchase|do(X=no_social)) = 0.5025
P(Y=no_purchase|do(X=social)) = 0.4525
P(Y=purchase|do(X=social)) = 0.5475

Causal Effects DataFrame:
           X            Y  P(Y|do(X))
0  no_social  no_purchase      0.4975
1  no_social     purchase      0.5025
2     social  no_purchase      0.4525
3     social     purchase      0.5475


In [4]:
# Calculate Average Treatment Effect (ATE)
print("\n=== Average Treatment Effect Analysis ===")

# Check if we have binary variables (works with string values too)
print(f"X values: {x_values}")
print(f"Y values: {y_values}")

# For binary treatment and outcome, calculate ATE
if len(x_values) == 2 and len(y_values) == 2:
    # Identify treatment and control values (assuming "social" is treatment, "purchase" is positive outcome)
    treatment_val = "social" if "social" in x_values else x_values[1]  # higher value
    control_val = "no_social" if "no_social" in x_values else x_values[0]  # lower value
    positive_outcome = "purchase" if "purchase" in y_values else y_values[1]
    negative_outcome = "no_purchase" if "no_purchase" in y_values else y_values[0]
    
    print(f"Treatment: {treatment_val}, Control: {control_val}")
    print(f"Positive outcome: {positive_outcome}, Negative outcome: {negative_outcome}")
    
    # Expected value of Y under intervention do(X=treatment)
    # E[Y|do(X=treatment)] = P(Y=positive|do(X=treatment)) * 1 + P(Y=negative|do(X=treatment)) * 0
    e_y_do_treatment = front_door_adjustment(treatment_val, positive_outcome)
    
    # Expected value of Y under intervention do(X=control)
    e_y_do_control = front_door_adjustment(control_val, positive_outcome)
    
    # Average Treatment Effect
    ate = e_y_do_treatment - e_y_do_control
    
    print(f"\nE[Y|do(X={treatment_val})] = {e_y_do_treatment:.4f}")
    print(f"E[Y|do(X={control_val})] = {e_y_do_control:.4f}")
    print(f"Average Treatment Effect (ATE) = {ate:.4f}")
    
    if ate > 0:
        print(f"Interpretation: {treatment_val} has a positive causal effect on {positive_outcome} (increase of {ate:.4f})")
    elif ate < 0:
        print(f"Interpretation: {treatment_val} has a negative causal effect on {positive_outcome} (decrease of {abs(ate):.4f})")
    else:
        print(f"Interpretation: {treatment_val} has no causal effect on {positive_outcome}")
else:
    print("ATE calculation requires binary X and Y variables")
    print("For non-binary variables, use the causal effects table above")

# Compare with naive association (without controlling for confounding)
print("\n=== Comparison with Naive Association ===")
naive_df = social_media_data.groupby(['X', 'Y']).size().reset_index(name='count')
naive_df['total_x'] = naive_df.groupby('X')['count'].transform('sum')
naive_df['p_y_given_x'] = naive_df['count'] / naive_df['total_x']

print("Naive P(Y|X) (ignoring confounding):")
print(naive_df[['X', 'Y', 'p_y_given_x']])

# Calculate naive effect for comparison
if len(x_values) == 2 and len(y_values) == 2:
    # Get naive probabilities
    naive_treatment = naive_df[(naive_df['X'] == treatment_val) & (naive_df['Y'] == positive_outcome)]
    naive_control = naive_df[(naive_df['X'] == control_val) & (naive_df['Y'] == positive_outcome)]
    
    if len(naive_treatment) > 0 and len(naive_control) > 0:
        naive_e_y_treatment = naive_treatment['p_y_given_x'].iloc[0]
        naive_e_y_control = naive_control['p_y_given_x'].iloc[0]
        naive_effect = naive_e_y_treatment - naive_e_y_control
        
        print(f"\nNaive effect (biased): {naive_effect:.4f}")
        print(f"Causal effect (unbiased): {ate:.4f}")
        print(f"Bias correction: {ate - naive_effect:.4f}")
        
        if abs(ate - naive_effect) > 0.01:
            print(f"⚠️  Significant bias detected! The unmeasured confounder is affecting the results.")
        else:
            print("✅ Bias is minimal in this case.")


=== Average Treatment Effect Analysis ===
X values: ['no_social', 'social']
Y values: ['no_purchase', 'purchase']
Treatment: social, Control: no_social
Positive outcome: purchase, Negative outcome: no_purchase

E[Y|do(X=social)] = 0.5475
E[Y|do(X=no_social)] = 0.5025
Average Treatment Effect (ATE) = 0.0450
Interpretation: social has a positive causal effect on purchase (increase of 0.0450)

=== Comparison with Naive Association ===
Naive P(Y|X) (ignoring confounding):
           X            Y  p_y_given_x
0  no_social  no_purchase       0.0975
1  no_social     purchase       0.9025
2     social  no_purchase       0.8525
3     social     purchase       0.1475

Naive effect (biased): -0.7550
Causal effect (unbiased): 0.0450
Bias correction: 0.8000
⚠️  Significant bias detected! The unmeasured confounder is affecting the results.
